### 1. Imports & Database Connection

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine

# Set plot visual style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Direct CSV file path (mounted inside container)
CSV_PATH = "/app/data/raw/Flight_Price_Dataset_of_Bangladesh.csv"

# Load dataset
df_raw = pd.read_csv(CSV_PATH)
print(f"Dataset Shape: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns")
df_raw.head()

### 2. Schema and Missing Values Profiling

In [ ]:
# 1. Check data types and non-null counts
print("--- Data Info ---")
df_raw.info()

# 2. Check exact missing/null value count per column
print("\n--- Missing Value Counts ---")
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw)) * 100
missing_df = pd.DataFrame({"Missing Rows": missing, "Percentage (%)": missing_pct})
display(missing_df[missing_df["Missing Rows"] > 0])

### 3. Data Integrity & Business Logic Checks

In [ ]:
# 1. Check for negative or zero fares
invalid_fares = df_raw[(df_raw["Base Fare (BDT)"] <= 0) | (df_raw["Total Fare (BDT)"] <= 0)]
print(f"Rows with invalid Base or Total Fare (<= 0): {len(invalid_fares)}")

# 2. Check if Total Fare == Base Fare + Tax & Surcharge
calculated_total = df_raw["Base Fare (BDT)"] + df_raw["Tax & Surcharge (BDT)"]
fare_discrepancies = np.abs(df_raw["Total Fare (BDT)"] - calculated_total) > 0.01
print(f"Rows with Total Fare discrepancies: {fare_discrepancies.sum()}")

# 3. Check unique airlines and seasonality labels
print("\nUnique Airlines:", df_raw["Airline"].unique())
print("Seasonality Classes:", df_raw["Seasonality"].unique())
print("Flight Classes:", df_raw["Class"].unique())

### 4. Statistical Distributions & Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot 1: Total Fare Distribution
sns.histplot(df_raw["Total Fare (BDT)"], bins=50, kde=True, ax=axes[0], color="teal")
axes[0].set_title("Total Fare Distribution (BDT)")
axes[0].set_xlabel("Fare (BDT)")

# Plot 2: Average Fare by Airline
airline_fares = df_raw.groupby("Airline")["Total Fare (BDT)"].mean().sort_values(ascending=False)
sns.barplot(x=airline_fares.values, y=airline_fares.index, ax=axes[1], palette="Blues_r")
axes[1].set_title("Average Total Fare by Airline")
axes[1].set_xlabel("Mean Fare (BDT)")

plt.tight_layout()
plt.show()

### 5. Prototype Cleaning Function

In [ ]:
def clean_flight_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Transforms and validates raw flight price data:
    - Trims strings and strips whitespace
    - Imputes or recalculates Total Fare
    - Parses timestamps and extracts clean flight date
    - Flags invalid negative fares
    """
    df = df.copy()
    
    # 1. Rename columns to standard snake_case
    df = df.rename(columns={
        "Airline": "airline",
        "Source": "source_code",
        "Source Name": "source_name",
        "Destination": "destination_code",
        "Destination Name": "destination_name",
        "Departure Date & Time": "departure_datetime",
        "Arrival Date & Time": "arrival_datetime",
        "Duration (hrs)": "duration_hrs",
        "Stopovers": "stopovers",
        "Aircraft Type": "aircraft_type",
        "Class": "flight_class",
        "Booking Source": "booking_source",
        "Base Fare (BDT)": "base_fare_bdt",
        "Tax & Surcharge (BDT)": "tax_and_surcharge_bdt",
        "Total Fare (BDT)": "total_fare_bdt",
        "Seasonality": "seasonality",
        "Days Before Departure": "days_before_departure"
    })
    
    # 2. String trimming
    str_cols = ["airline", "source_code", "source_name", "destination_code", 
                "destination_name", "stopovers", "flight_class", "seasonality"]
    for col in str_cols:
        df[col] = df[col].astype(str).str.strip()
        
    # 3. Handle missing/null fares and enforce calculation
    df["base_fare_bdt"] = pd.to_numeric(df["base_fare_bdt"], errors="coerce").fillna(0.0)
    df["tax_and_surcharge_bdt"] = pd.to_numeric(df["tax_and_surcharge_bdt"], errors="coerce").fillna(0.0)
    df["total_fare_bdt"] = df["base_fare_bdt"] + df["tax_and_surcharge_bdt"]
    
    # 4. Filter out invalid rows (negative/zero base fares)
    df = df[df["base_fare_bdt"] > 0]
    
    # 5. Parse Datetime & extract date
    df["departure_datetime"] = pd.to_datetime(df["departure_datetime"])
    df["arrival_datetime"] = pd.to_datetime(df["arrival_datetime"])
    df["flight_date"] = df["departure_datetime"].dt.date
    
    # 6. Construct Route Helper Fields
    df["route_code"] = df["source_code"] + " -> " + df["destination_code"]
    df["route_name"] = df["source_name"] + " -> " + df["destination_name"]
    
    return df

# Test the prototype
df_clean = clean_flight_data(df_raw)
print(f"Cleaned Data Count: {len(df_clean):,} records")
df_clean.head(3)